In [1]:
import pandas as pd
import requests
import httpx
import json
import os
from qdrant_client import QdrantClient

from langfuse import Langfuse
from langfuse.decorators import observe, langfuse_context

/home/jeaneudes/Desktop/project_code/rag_venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# SET UP

In [2]:
# Load configuration from the config-template file
with open('/langfuse_trials/config-template.json', 'r') as config_file:
    config_dict = json.load(config_file)

# Extract API URLs and API Keys from the configuration
nlp_api_key = config_dict.get("APP_SERVICE_NLP_API_KEY")
api_url_base = config_dict.get("APP_SERVICE_NLP_API_URL")
vec_db_api_key = config_dict.get("APP_SERVICE_VEC_DB_API_KEY")

#LANGFUSE API KEYS
langfuse_public_api_key = config_dict.get("LANGFUSE_PUBLIC_KEY")
langfuse_private_api_key = config_dict.get("LANGFUSE_SECRET_KEY")

# Define Model and API URL
llm_model_id = "anthropic.claude-3-5-sonnet-20240620-v1:0"
llm_model_api_url = f"{api_url_base}/model/{llm_model_id}/invoke"

embedding_model_id = "amazon.titan-embed-text-v2:0"
embedding_api_url = f"{api_url_base}/model/{embedding_model_id}/invoke"
api_url = f"{api_url_base}:443"

headers = {  
    'Content-Type': 'application/json',  
    "x-api-key": nlp_api_key,
    'openai-standard': 'True'
    }

In [ ]:
# Create an HTTP client with necessary headers
http_client = httpx.Client(headers={"llm-traces": "true"})

#You can set the credentials either as environment variables or constructor arguments
os.environ["LANGFUSE_PUBLIC_KEY"] = langfuse_public_api_key
os.environ["LANGFUSE_SECRET_KEY"] = langfuse_private_api_key
os.environ["LANGFUSE_HOST"] = api_url_base

langfuse = Langfuse()

In [ ]:
# Configure the Langfuse context
langfuse_context.configure(
    secret_key=os.getenv("LANGFUSE_SECRET_KEY"),
    public_key=os.getenv("LANGFUSE_PUBLIC_KEY"),
    httpx_client=http_client,
    host=os.getenv("LANGFUSE_HOST")
 	# debug = False Prints debug logs to the console
 	# threads = 1 Specifies the number of consumer threads to execute network requests to the Langfuse server.
 	# max_retries = 3 Specifies the number of times the SDK should retry network requests for tracing.
	# timeout = 20 Timeout in seconds for network requests
	# sample_rate = 1.0 Control the volume of traces collected by the Langfuse server.
)

# One event trace

Creating a Trace

To create a trace, you can use the @observe decorator in Langfuse, which will automatically capture the duration, nesting, function name, input, and output of the trace.

In [8]:
@observe()
def call_claude(system_prompt, user_prompt) -> str:
    """
    Calls the LLM API with the given system and user prompts to obtain a response.

    Returns:
        str: The generated response from the LLM API.
    """
    payload = {
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "temperature": 0,
        "max_tokens": 4096
    }
    
    response = requests.post(            
        url=llm_model_api_url,
        headers=headers,
        verify=False,
        json=payload,
    )

    response.raise_for_status()

    response_data = response.json()['choices'][0]['message']['content']

    return response_data

In [9]:
system_prompt = "Give me 5 alternatives questions to the question I give you"
user_prompt = "What is the price of Erbitux?"
call_claude(system_prompt, user_prompt)

/home/jeaneudes/Desktop/project_code/rag_venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.nlp.dev.uptimize.merckgroup.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


'Here are 5 alternative questions related to the price of Erbitux:\n\n1. How much does a typical course of Erbitux treatment cost?\n\n2. What factors influence the pricing of Erbitux in different countries?\n\n3. Are there any generic alternatives to Erbitux, and how do their prices compare?\n\n4. How has the price of Erbitux changed over the past decade?\n\n5. What insurance coverage options are available for patients prescribed Erbitux, and how does this affect out-of-pocket costs?'

# Multiple event trace + update

Traces can be dynamically updated with metadata such as user ID, session ID, tags, and trace names.

In [10]:
collection_name = 'collection_name'
k_chunks_retrieved = 3

@observe()
def call_embedding_api(text):
    """
    Calls the embedding API to generate embeddings for the given text.
    We need to format the input to be accepted by the API.

    Returns:
        list: A list of floats representing the embedding vector.
"""
    payload = {"input": text}
    response = requests.request("POST", embedding_api_url, headers=headers, data=json.dumps(payload))
    response.raise_for_status()  # This will raise an exception for HTTP errors
    response_data = response.json()['data'][0]['embedding']
    return response_data

@observe()
def call_llm_api(system_prompt: str, user_prompt: str) -> str:
    """
    Calls the LLM API with the given system and user prompts to obtain a response.

    Returns:
        str: The generated response from the LLM API.
    """
    payload = {
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "temperature": 0,
        "max_tokens": 4096
    }
    
    response = requests.post(            
        url=llm_model_api_url,
        headers=headers,
        verify=False,
        json=payload,
    )

    response.raise_for_status()

    response_data = response.json()['choices'][0]['message']['content']
    return response_data

@observe()
def query_qdrant(question_embedding):
    """
    Queries the Qdrant database using the provided embedding.

    Returns:
        tuple: A tuple containing relevant chunks and metadata.
    """
    # Open the Qdrant connection
    qdrant_client = QdrantClient(
        url=api_url,
        api_key=vec_db_api_key,
    )

    # Query Qdrant
    search_result = qdrant_client.query_points(
        collection_name=collection_name,
        query=question_embedding,
        limit=k_chunks_retrieved
    )

    # Extract relevant chunks and metadata
    relevant_chunks = [hit.payload['text'] for hit in search_result.points]

    if qdrant_client:
        qdrant_client.close()
        
    return relevant_chunks

@observe()
def process_question(question: str) -> list:
    question_embedding = call_embedding_api(question)
    relevant_chunks = query_qdrant(question_embedding)
    langfuse_context.update_current_trace(
    user_id="7891011",
    session_id="HIJKLMN",
    tags=["erbitux", "tag_2"]
    )

    return relevant_chunks

#process_question("What is the price of Erbitux ?")


: 

# Scoring in Langfuse

You can integrate your evaluation process within Langfuse


- LLM-as-a-judge: Fully managed evaluators run on production or development traces within Langfuse
- User feedback: Collect feedback from your users and add it to traces in Langfuse
- Manual labeling: Annotate traces with human feedback in managed workflows
- Custom: Build your own evaluation pipelines via Langfuse APIs/SDKs for full flexibility

You can either:

- Use custom evaluation functions: Implement your own evaluation logic and add scores based on the result.
- Leverage model-based evaluation: Set up LLM Traces to automatically evaluate outputs based on pre-defined metrics or model judgments.

Key Steps in Building an External Evaluation Pipeline

- Fetch Your Traces: Retrieve your application traces from LLM Traces for evaluation.
- Run Your Evaluations: Apply any custom evaluation logic you prefer.
- Save Your Results: Attach the evaluations as scores back to the original traces.

In [ ]:
# Generate traces with the function process_question

file_path = "generated_alternative_questions.json"
with open(file_path, 'r') as file:
    data = json.load(file)
# Iterate through the keys and print them
for key in data.keys():
    process_question(key)

In [ ]:
# Retrieving the generated traces based on multiple characteristics

from datetime import datetime, timedelta

BATCH_SIZE = 10
TOTAL_TRACES = 50

# now = datetime.now()
# five_am_today = datetime(now.year, now.month, now.day, 5, 0)
# five_am_yesterday = five_am_today - timedelta(days=1)

traces_batch = langfuse.fetch_traces(page=1,
                                     limit=BATCH_SIZE,
                                    # tags="tag1",
                                    # from_timestamp=five_am_yesterday,
                                    # to_timestamp=datetime.now()
                                    ).data

print(f"Traces in first batch: {len(traces_batch)}")

Traces in first batch: 10


In [65]:
traces_batch[0]

TraceWithDetails(id='619d7c9a-305c-41cb-b2de-08dbc727a678', timestamp=datetime.datetime(2025, 5, 13, 11, 41, 40, 338000, tzinfo=datetime.timezone.utc), name='process_question', input={'args': ['皮膚症状の対処法は？'], 'kwargs': {}}, output=['アービタックス投与でみられる皮膚症状には外用のステロイド剤や抗生剤で対処をお願いいたします。詳細は「注意すべき皮膚症状とその対策」をご確認ください。1)\nまた、G3以上の皮膚症状発現時には、電子添文の用量調節の目安にしたがい、休薬、減量をお願いいたします。', '## ●患者指導\n\nア-ビタックス投与中の患者は、 ⽪膚のバリア機能が低下していると考えられるため、 ⽇常での⽪膚 ケアが重要になります。\n\n以下に、 患者に指導すべきポイントを⽰します。\n\n## ⽪膚ケアの指導ポイント\n\n- ·肌を清潔に保つ。\n- ·低刺激性で⾹料、 保存料を含有しない⽯鹸を使⽤する。\n- ·シャワ-はぬるま湯で使⽤し、 ⻑いシャワ-、 熱いシャワ-は避ける。\n- ·シャワ-または⼊浴後、 乾燥している部位になるべく早く保湿剤を塗る。\n- ·直射⽇光を避け、 紫外線の防⽌効果が⾼い⽇焼け⽌めを⼗分に使⽤する。\n- ·外出する時は、 広いつばのある帽⼦を着⽤する。\n- ·締め付け度の強い下着は着⽤しない。\n- ·底が固い靴、 幅の狭い靴は履かない。 サイズの合った柔らかい靴を履く。\n- ·化粧品 （基礎及びメイクアップ⽤） は保湿性が⾼く、 刺激の少ないものを選ぶ。\n\n## 市販医薬品·化粧品の使⽤について\n\n- ·⽪膚に刺激のある薬剤及び化粧品は症状を悪化させる可能性があるため、市販医薬品や刺激の 強い化粧品の使⽤は控えるように指導する。\n\n⽪膚症状に改善が⾒られた際にも、⾃⼰判断で⽪膚ケア·治療を中断しないよう指導をお願い します。\n\n詳細は別冊⼦、ア-ビタックス ® 注射液100mg,500mg 注意すべき⽪膚症状とその対策』 をご覧下さい。 『', '作⽤\n\n## ●対処法

In [ ]:
# Creating simple prompt to evaluate length of the question and evaluating one trace

def call_llm_api_and_get_score(system_prompt: str, user_prompt: str, text: str) -> str:
    """
    Calls the LLM API with the given system and user prompts to obtain a response.

    Returns:
        str: The generated response from the LLM API.
    """
    payload = {
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt.format(text=text)},
        ],
        "temperature": 0,
        "max_tokens": 4096
    }
    
    response = requests.post(            
        url=llm_model_api_url,
        headers=headers,
        verify=False,
        json=payload,
    )

    response.raise_for_status()

    response_data = response.json()['choices'][0]['message']['content']
    print(response_data)
    return response_data

system_prompt = """

Count the number of character of the question you will receive. If it is a good sentiment answer says "good" if it is a bad sentiment answer says "bad"
"""
user_prompt = """
This is the question:
{text}
"""

# call_llm_api_and_get_score(system_prompt, user_prompt, traces_batch[0])

/home/jeaneudes/Desktop/project_code/rag_venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.nlp.dev.uptimize.merckgroup.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


too short


'too short'

In [79]:
import math

for page_number in range(1, math.ceil(TOTAL_TRACES / BATCH_SIZE)):
    traces_batch = langfuse.fetch_traces(page=1,
                                     limit=BATCH_SIZE,
                                     tags="tag1",
                                    # from_timestamp=five_am_yesterday,
                                    # to_timestamp=datetime.now()
                                    ).data

    for trace in traces_batch:
        if trace.output is None:
            print(f"Trace {trace.name} had no output, skipping")
            continue

        langfuse.score(
            trace_id=trace.id,
            name="size_score",
            value=call_llm_api_and_get_score(system_prompt, user_prompt, trace)
        )

    print(f"Batch {page_number} processed")

/home/jeaneudes/Desktop/project_code/rag_venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.nlp.dev.uptimize.merckgroup.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


too short


/home/jeaneudes/Desktop/project_code/rag_venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.nlp.dev.uptimize.merckgroup.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


too long


/home/jeaneudes/Desktop/project_code/rag_venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.nlp.dev.uptimize.merckgroup.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


...

too long


/home/jeaneudes/Desktop/project_code/rag_venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.nlp.dev.uptimize.merckgroup.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


too long


/home/jeaneudes/Desktop/project_code/rag_venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.nlp.dev.uptimize.merckgroup.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


too long
Batch 4 processed


In [ ]:
traces_batch = langfuse.fetch_trace(page=1,
                                     limit=BATCH_SIZE,
                                     tags="tag1",
                                    # from_timestamp=five_am_yesterday,
                                    # to_timestamp=datetime.now()
                                    ).data

langfuse_context.score_current_trace(
    name="feedback-on-trace",
    value=1,
    comment="This response is very weird, please check what is happening."
)

Failed to score observation: No trace found in the current context


# Datasets

In [ ]:
# Test of similarity score on GT answer and LLM generated answer
questions_df = pd.read_excel("gt_dataset.xlsx")

langfuse.create_dataset(name="gt_dataset")

for index, item in questions_df.iterrows():
    langfuse.create_dataset_item(
        dataset_name="gt_dataset",
        input=item['Answer'],  # Store the 'Answer' as input
        expected_output=item["LLM_Answer"]  # Store the 'LLM_Answer' as expected output
    )

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def get_cosine_similarity(experiment_name):
    """
    Runs the experiment using Langfuse and evaluates similarity scores.

    Args:
        experiment_name (str): Name of the experiment.

    Returns:
        None
    """
    dataset = langfuse.get_dataset("gt_dataset")
    
    for item in dataset.items:
        # Access the 'input' and 'expected_output' fields
        answer = item.input
        llm_answer = item.expected_output

        # Observe each item and link it to the experiment run
        with item.observe(run_name=experiment_name) as trace_id:
            answer_embedding = call_embedding_api(answer)
            llm_answer_embedding = call_embedding_api(llm_answer)
            similarity_score = cosine_similarity([answer_embedding], [llm_answer_embedding])[0][0]
            
            # Log similarity score to Langfuse
            langfuse.score(
                trace_id=trace_id,
                name="similarity",
                value=similarity_score
            )
            print(langfuse.score)

get_cosine_similarity("experiment_1")

<bound method Langfuse.score of <langfuse.client.Langfuse object at 0x7902f8be6c00>>
<bound method Langfuse.score of <langfuse.client.Langfuse object at 0x7902f8be6c00>>
<bound method Langfuse.score of <langfuse.client.Langfuse object at 0x7902f8be6c00>>
<bound method Langfuse.score of <langfuse.client.Langfuse object at 0x7902f8be6c00>>
<bound method Langfuse.score of <langfuse.client.Langfuse object at 0x7902f8be6c00>>
<bound method Langfuse.score of <langfuse.client.Langfuse object at 0x7902f8be6c00>>
<bound method Langfuse.score of <langfuse.client.Langfuse object at 0x7902f8be6c00>>
<bound method Langfuse.score of <langfuse.client.Langfuse object at 0x7902f8be6c00>>
<bound method Langfuse.score of <langfuse.client.Langfuse object at 0x7902f8be6c00>>
<bound method Langfuse.score of <langfuse.client.Langfuse object at 0x7902f8be6c00>>
<bound method Langfuse.score of <langfuse.client.Langfuse object at 0x7902f8be6c00>>
<bound method Langfuse.score of <langfuse.client.Langfuse object 